# prediction.ipynb: Project Workflow (Predict Churn for a New Customer)

## 1. Objective
Use the **already trained** churn ANN to predict whether a **new customer** is likely to leave the bank. Nothing is trained or fitted here. The notebook only loads saved files, prepares the new customer's data the same way the training data was prepared, and runs the model.

- **Input:** one customer described by 10 raw values (credit score, country, gender, age, and so on)
- **Output:** a churn probability between 0 and 1, then a decision ("likely" or "not likely" to churn)

## 2. Files this notebook needs
These are created by the training notebook (experiments.ipynb) and must be in the same folder:

| File | Contents | Used for |
|---|---|---|
| `model.h5` | Trained ANN (architecture + weights) | Making the prediction |
| `onehot_encoder_geo.pkl` | Fitted OneHotEncoder for `Geography` | Turning the country into 3 columns |
| `label_encoder_gender.pkl` | Fitted LabelEncoder for `Gender` | Turning Male/Female into 1/0 |
| `scaler.pkl` | Fitted StandardScaler | Scaling the features like the training data |

## 3. Workflow at a glance

```
model.h5 + onehot_encoder_geo.pkl + label_encoder_gender.pkl + scaler.pkl
        |
        v
[1] Import libraries (Step 1)
        |
        v
[2] Load the model, encoders and scaler (Step 2)
        |
        v
[3] Define the new customer as a dictionary (Step 3)
        |
        v
[4] One-hot encode Geography -> 3 columns (Step 4)
        |
        v
[5] Convert the dictionary to a one-row DataFrame (Step 5)
        |
        v
[6] Encode Gender (Male -> 1) (Step 6)
        |
        v
[7] Join the one-hot columns -> 12 features in training order (Step 7)
        |
        v
[8] Scale with the saved scaler (Step 8)
        |
        v
[9] model.predict -> churn probability (Steps 9-11)
        |
        v
[10] Apply the 0.5 threshold -> final message (Step 12)
```

## 4. Stage-by-stage details

### Stage 1: Imports and loading (Steps 1-2)
- Import `tensorflow`/`load_model`, `pickle`, `pandas` and `numpy`.
- Load the model with `load_model('model.h5')` and the three `.pkl` files with `pickle.load`.
- The one-hot encoder is stored in a variable named `label_encoder_geo`, but it is actually a `OneHotEncoder`.

### Stage 2: Defining the new customer (Step 3)
`input_data` is a dictionary with the same 10 raw features the model was trained on:

| Feature | Example value |
|---|---|
| CreditScore | 600 |
| Geography | France |
| Gender | Male |
| Age | 40 |
| Tenure | 3 |
| Balance | 60,000 |
| NumOfProducts | 2 |
| HasCrCard | 1 |
| IsActiveMember | 1 |
| EstimatedSalary | 50,000 |

Change these values to test other customers.

### Stage 3: Preprocessing the input (Steps 4-8)
The new data must go through the **same transformations as the training data**, using the saved objects. Only `transform` is used, never `fit`, so the mappings learned during training are reused.
1. **Geography:** France becomes `[1, 0, 0]` (`Geography_France`, `Geography_Germany`, `Geography_Spain`).
2. **DataFrame:** the dictionary becomes a one-row DataFrame.
3. **Gender:** Male becomes 1.
4. **Combine:** the text `Geography` column is dropped and the three one-hot columns are added, giving 12 columns. The **column order must match the training data**, because the scaler and model work by position.
5. **Scale:** `scaler.transform(...)` standardises the 12 values using the training mean and standard deviation.

### Stage 4: Prediction (Steps 9-11)
- `model.predict(input_scaled)` returns a 2-D array holding the churn probability.
- `prediction[0][0]` extracts the single number.
- For the example customer the probability is about **0.0297** (roughly a 3% chance of churn).

### Stage 5: Decision (Step 12)
- If the probability is **greater than 0.5**, the customer is "likely to churn".
- Otherwise, the customer is "not likely to churn".
- The example customer gives: **"The customer is not likely to churn."**

## 5. How to run this notebook
1. Make sure `model.h5`, `onehot_encoder_geo.pkl`, `label_encoder_gender.pkl` and `scaler.pkl` exist in the folder. If not, run the training notebook first.
2. Edit the values in `input_data` (Step 3) to describe the customer you want to test.
3. Run the cells from top to bottom and read the final message.

## 6. Common problems
- **`FileNotFoundError`:** one of the four saved files is missing from the folder.
- **Wrong predictions or errors about feature names/count:** the 12 columns are not in the same order as in training, or a transformation was skipped.
- **Warnings about feature names or the HDF5 format:** harmless here, the result is still correct.

## 7. Libraries used
`tensorflow` / `keras` (`load_model`), `pickle`, `pandas`, `numpy`.

### Step 1: Import the libraries
- `tensorflow` and `load_model`: load the neural network that was trained and saved earlier.
- `pickle`: loads the saved encoders and scaler.
- `pandas` (`pd`): builds DataFrames for the new customer's data.
- `numpy` (`np`): numerical array support.

The TensorFlow deprecation warning in the output is harmless.

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

### Step 2: Load the trained model, encoders and scaler
This notebook only **predicts**; nothing is retrained. It reloads the artifacts saved by the training notebook:

- `load_model('model.h5')`: the trained ANN (architecture + weights).
- `onehot_encoder_geo.pkl`: the fitted one-hot encoder for `Geography`. (It is stored in the variable `label_encoder_geo`, but it is actually a `OneHotEncoder`.)
- `label_encoder_gender.pkl`: the fitted label encoder for `Gender`.
- `scaler.pkl`: the fitted `StandardScaler`.

Using the saved objects guarantees the new data is transformed in exactly the same way as the training data.

In [2]:
### Load the trained model, scaler pickle,onehot
model=load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

### Step 3: Define the new customer's data
`input_data` is a dictionary describing one customer (a person whose churn we want to predict). It contains the same 10 raw features the model was trained on: credit score 600, lives in France, male, age 40, tenure 3 years, balance 60,000, 2 products, has a credit card, is an active member, estimated salary 50,000.

Change the values here to test other customers.

In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

### Step 4: One-hot encode `Geography`
- `transform([[input_data['Geography']]])` converts "France" into the one-hot vector `[1, 0, 0]`. Only `transform` is used (not `fit_transform`) because the encoder has already learned the categories.
- `.toarray()` turns the sparse result into a normal array.
- A DataFrame with columns `Geography_France`, `Geography_Germany`, `Geography_Spain` is created from it.

The `UserWarning` appears because the encoder was trained with named columns and is now given a plain list; the result is still correct.

In [4]:
# One-hot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


### Step 5: Convert the input dictionary to a DataFrame
`pd.DataFrame([input_data])` turns the dictionary into a single-row DataFrame. The scaler and model expect tabular data with named columns, not a dictionary.

In [5]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


### Step 6: Encode `Gender`
`label_encoder_gender.transform(...)` converts the text "Male" into its number (**1**), matching the encoding used during training. It uses `transform` only, so the mapping learned from the training data is applied unchanged.

In [6]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


### Step 7: Combine the one-hot columns with the other features
Drops the text `Geography` column and appends the three one-hot columns (`axis=1` = side by side). The DataFrame now has the same 12 columns, in the same order, as the training data. Matching the column order is essential because the model and scaler work on positions.

In [7]:
## concatination one hot encoded
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


### Step 8: Scale the input data
`scaler.transform(input_df)` standardises every feature with the **mean and standard deviation from the training data**. The result is a NumPy array with one row of 12 scaled values, which is the format the model expects.

In [9]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [10]:
import pandas as pd
pd.DataFrame(input_scaled)

,0,1,2,3,4,5,6,7,8,9,10,11
0,-0.535985,0.913248,0.104794,-0.695393,-0.257811,0.808436,0.649203,0.974817,-0.876832,1.001501,-0.579467,-0.576388


### Step 9: Predict churn
`model.predict(input_scaled)` runs the scaled customer data through the network. The output is a 2-D array holding the **probability of churn** (between 0 and 1). Here it is roughly 0.03, i.e. about a 3% chance that the customer leaves.

In [11]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step


array([[0.05241786]], dtype=float32)

### Step 10: Extract the probability value
`prediction[0][0]` pulls the single number out of the 2-D array (first row, first column) so it can be compared with a threshold.

In [12]:
prediction_proba = prediction[0][0]

### Step 11: Display the probability
Shows the extracted value (about 0.0297). Values close to 0 mean the customer is likely to stay; values close to 1 mean the customer is likely to churn.

In [13]:
  prediction_proba

np.float32(0.052417863)

### Step 12: Turn the probability into a decision
A threshold of **0.5** is applied: if the churn probability is greater than 0.5 the customer is classified as likely to churn, otherwise as not likely to churn.

Here the probability is about 0.03, so the output is **"The customer is not likely to churn."**

In [14]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
